# Neuroscience-Inspired Memory Replay for Continual Learning

Reproduces experiments from:
> Nalagatla & Grandhe (2025) **"Neuroscience-Inspired Memory Replay for Continual Learning: A Comparative Study of Predictive Coding and Backpropagation-Based Strategies"** — [arXiv:2512.00619](https://arxiv.org/abs/2512.00619)

Compares two generative-replay strategies for continual learning on Split-MNIST:

| Strategy | Generator | Learning rule |
|----------|-----------|---------------|
| **PC Replay** | 4-layer predictive coding network (FabricPC / JAX) | Local Hebbian — inference + weight update |
| **VAE Replay** | Variational autoencoder (Flax / backprop) | ELBO — reconstruction + KL divergence |

### Continual learning setup
- **Split-MNIST**: 5 sequential tasks (digits 0–1, 2–3, 4–5, 6–7, 8–9)
- After each task the generator for that task is frozen; subsequent tasks use replay from all previous generators
- Mixing ratio β = 0.5 (equal real and replay samples per batch)

### Paper reference results
| Method | Avg Accuracy | Forgetting |
|--------|-------------|------------|
| PC Replay | **94.20 %** | **2.10 %** |
| VAE Replay | **89.10 %** | **5.80 %** |

> **Quick mode** (this notebook default) uses 10 generator epochs and 20 classifier epochs — much less than the paper's 50. Use `quick=False` for paper-comparable numbers (takes ~30 min on CPU).

## 0. Imports & Setup

In [ ]:
import sys, os, pathlib
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import jax

# ── Locate the project root (directory that contains fabricpc/) ──────────────
_cwd = pathlib.Path.cwd()
_proj_root = None
for _candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_candidate / 'fabricpc').exists():
        _proj_root = _candidate
        break
if _proj_root is None:
    raise RuntimeError(f"Could not find FabricPC project root from {_cwd}")
if str(_proj_root) not in sys.path:
    sys.path.insert(0, str(_proj_root))

# ── Locate continual_learning_replay.py ──────────────────────────────────────
_py_candidates = [
    _cwd / 'continual_learning_replay.py',
    _proj_root / 'examples' / 'continual_learning_replay.py',
]
_py_path = next((p for p in _py_candidates if p.exists()), None)
if _py_path is None:
    raise FileNotFoundError(
        f"continual_learning_replay.py not found; tried: {_py_candidates}"
    )

# ── Import the module ─────────────────────────────────────────────────────────
# The module must be registered in sys.modules BEFORE exec_module so that
# Flax's nn.Module metaclass can resolve cls.__module__ during class creation.
import importlib.util
_MOD_NAME = 'continual_learning_replay'
_spec = importlib.util.spec_from_file_location(_MOD_NAME, _py_path)
clr = importlib.util.module_from_spec(_spec)
sys.modules[_MOD_NAME] = clr          # ← register first, then execute
_spec.loader.exec_module(clr)

from fabricpc import setup_jax
setup_jax()
jax.config.update('jax_default_prng_impl', 'threefry2x32')

print(f'Project root: {_proj_root}')
print(f'Module path : {_py_path}')
print(f'JAX devices : {jax.devices()}')
print(f'NumPy       : {np.__version__}')


## 1. Data Loading and Task Splits

MNIST is split into 5 tasks of 2 classes each. `create_split_tasks` returns a list of `(x_train, y_train, x_test, y_test)` tuples, one per task.

In [ ]:
SEED   = 42
N_TASKS = 5

print('Loading MNIST ...')
x_tr, y_tr, x_te, y_te = clr.load_mnist()
print(f'  Train: {len(x_tr)}  Test: {len(x_te)}  dim={x_tr.shape[1]}')

tasks = clr.create_split_tasks(
    x_tr, y_tr, x_te, y_te,
    n_tasks=N_TASKS,
    classes_per_task=2,
    seed=SEED,
)

print(f'\nTask splits:')
for i, (xtr, ytr, xte, yte) in enumerate(tasks):
    classes = sorted(np.unique(ytr).tolist())
    print(f'  Task {i+1}: classes {classes}  '
          f'  train={len(xtr)}  test={len(xte)}')

### Sample images from each task

In [ ]:
fig, axes = plt.subplots(N_TASKS, 6, figsize=(9, N_TASKS * 1.5))
rng = np.random.default_rng(SEED)

for task_idx, (xtr, ytr, _, _) in enumerate(tasks):
    classes = sorted(np.unique(ytr).tolist())
    for col in range(6):
        c = classes[col % len(classes)]
        idx = rng.choice(np.where(ytr == c)[0])
        axes[task_idx, col].imshow(xtr[idx].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[task_idx, col].axis('off')
    axes[task_idx, 0].set_ylabel(f'Task {task_idx+1}\n({classes})', fontsize=9, rotation=90)

fig.suptitle('MNIST task splits — 6 samples per task', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 2. Run Experiment

Set `quick=True` for a fast smoke test (10 gen epochs, 20 clf epochs).  
Set `quick=False` to match the paper's 50 epochs per task (much slower on CPU).

In [ ]:
QUICK   = True    # False → 50 epochs/task, paper-comparable
N_TRIALS = 1      # paper uses 5; increase for error bars

cfg = clr.DATASET_CONFIGS['mnist']

epochs_gen = cfg['quick_epochs_gen'] if QUICK else cfg['full_epochs_gen']
epochs_clf = cfg['quick_epochs_clf'] if QUICK else cfg['full_epochs_clf']

print(f'Mode      : {"QUICK" if QUICK else "FULL"}')
print(f'Gen epochs: {epochs_gen}/task')
print(f'Clf epochs: {epochs_clf}/task')
print(f'Trials    : {N_TRIALS}')
print()

results = {}

for method in ['pc', 'vae']:
    print(f'--- Method: {method.upper()} ---')
    trial_metrics  = []
    trial_matrices = []

    for trial in range(N_TRIALS):
        trial_seed = SEED + trial * 1000
        t0 = time.time()
        acc_mat = clr.run_experiment(
            tasks=tasks,
            method=method,
            data_dim=784,
            n_classes=10,
            epochs_gen=epochs_gen,
            epochs_clf=epochs_clf,
            mix_ratio=0.5,
            batch_size=128,
            gen_latent_dim=cfg['latent_dim'],
            gen_hidden_dim=cfg['hidden_dim'],
            clf_hidden_dim=400,
            gen_samples=cfg['gen_samples_per_task'],
            lr=1e-3,
            seed=trial_seed,
            verbose=False,
        )
        elapsed = time.time() - t0
        m = clr.compute_metrics(acc_mat)
        trial_metrics.append(m)
        trial_matrices.append(acc_mat)
        print(f'  Trial {trial+1}/{N_TRIALS}: '
              f'Avg Acc={m["avg_accuracy"]*100:.1f}%  '
              f'Forgetting={m["avg_forgetting"]*100:.1f}%  '
              f'({elapsed:.0f}s)')

    # Aggregate
    keys = list(trial_metrics[0].keys())
    agg = {}
    for k in keys:
        vals = [m[k] for m in trial_metrics]
        agg[k]        = float(np.mean(vals))
        agg[k+'_std'] = float(np.std(vals, ddof=0))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        agg['acc_matrix']     = np.nanmean(trial_matrices, axis=0)
        agg['acc_matrix_std'] = np.nanstd(trial_matrices, axis=0, ddof=0)
    results[method] = agg

print('\nDone.')

## 3. Accuracy Matrix Heatmaps

Each cell `(i, j)` shows accuracy on **task j** immediately after training on **task i**.  
The diagonal (current task) should be near 100 %. Off-diagonal entries below the diagonal reveal forgetting.

In [ ]:
def plot_acc_matrix(ax, mat, title, n_tasks):
    """Plot an accuracy matrix as a masked heatmap."""
    # Mask upper triangle (not yet evaluated)
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    display = np.where(mask, np.nan, mat)

    cmap = plt.cm.RdYlGn
    cmap.set_bad(color='#f0f0f0')  # grey for masked cells

    im = ax.imshow(display, vmin=0, vmax=1, cmap=cmap, aspect='auto')

    for i in range(n_tasks):
        for j in range(n_tasks):
            if not mask[i, j]:
                val = mat[i, j]
                color = 'white' if val < 0.4 or val > 0.85 else 'black'
                ax.text(j, i, f'{val*100:.1f}', ha='center', va='center',
                        fontsize=9, color=color)

    ax.set_xticks(range(n_tasks))
    ax.set_yticks(range(n_tasks))
    ax.set_xticklabels([f'T{i+1}' for i in range(n_tasks)])
    ax.set_yticklabels([f'After T{i+1}' for i in range(n_tasks)])
    ax.set_xlabel('Evaluated on task')
    ax.set_title(title, fontsize=11, pad=8)
    return im


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

im_pc  = plot_acc_matrix(axes[0], results['pc']['acc_matrix'],
                          'PC Replay — accuracy matrix', N_TASKS)
im_vae = plot_acc_matrix(axes[1], results['vae']['acc_matrix'],
                          'VAE Replay — accuracy matrix', N_TASKS)

fig.colorbar(im_pc,  ax=axes[0], fraction=0.046, pad=0.04, label='Accuracy')
fig.colorbar(im_vae, ax=axes[1], fraction=0.046, pad=0.04, label='Accuracy')
fig.suptitle('Split-MNIST continual learning — accuracy matrices', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Forgetting Curves

How quickly does the model forget earlier tasks as new tasks are introduced?  
Each line tracks accuracy on one task over training time.

In [ ]:
def plot_forgetting_curves(axes, mat, title, n_tasks):
    colors = plt.cm.tab10(np.linspace(0, 0.9, n_tasks))
    for task_idx in range(n_tasks):
        # Accuracy on task_idx at each evaluation step t >= task_idx
        x_vals, y_vals = [], []
        for t in range(task_idx, n_tasks):
            if not np.isnan(mat[t, task_idx]):
                x_vals.append(t + 1)
                y_vals.append(mat[t, task_idx] * 100)
        axes.plot(x_vals, y_vals, 'o-', color=colors[task_idx],
                  label=f'Task {task_idx+1}', linewidth=2, markersize=6)

    axes.set_xlabel('Training completed up to task')
    axes.set_ylabel('Accuracy on task (%)')
    axes.set_title(title, fontsize=11)
    axes.set_ylim(0, 105)
    axes.set_xticks(range(1, n_tasks + 1))
    axes.legend(loc='lower left', fontsize=8)
    axes.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_forgetting_curves(axes[0], results['pc']['acc_matrix'],
                       'PC Replay — forgetting curves', N_TASKS)
plot_forgetting_curves(axes[1], results['vae']['acc_matrix'],
                       'VAE Replay — forgetting curves', N_TASKS)
fig.suptitle('Split-MNIST — accuracy retention across tasks', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Metrics Summary and Paper Comparison

In [ ]:
# Paper reference values (arXiv:2512.00619, Table 1)
PAPER = {
    'pc' : {'avg_accuracy': 0.942, 'avg_forgetting': 0.021,
            'fwd_transfer': None,  'bwd_transfer': None},
    'vae': {'avg_accuracy': 0.891, 'avg_forgetting': 0.058,
            'fwd_transfer': None,  'bwd_transfer': None},
}

def pct(v, std=None):
    if v is None:
        return 'N/A'
    s = f'{v*100:.2f}%'
    if std is not None and std > 0:
        s += f' ±{std*100:.2f}%'
    return s

header = f"{'Method':<22} {'Avg Acc':>12} {'Forgetting':>12} {'Fwd BT':>10} {'Bwd BT':>10}"
print(header)
print('-' * len(header))

for method_label, key in [('PC Replay (ours)', 'pc'), ('VAE Replay (ours)', 'vae')]:
    r = results[key]
    print(f"{method_label:<22}"
          f" {pct(r['avg_accuracy'], r['avg_accuracy_std']):>12}"
          f" {pct(r['avg_forgetting'], r['avg_forgetting_std']):>12}"
          f" {pct(r['fwd_transfer'], r['fwd_transfer_std']):>10}"
          f" {pct(r['bwd_transfer'], r['bwd_transfer_std']):>10}")

print()
print('Paper reference (arXiv:2512.00619, Split-MNIST):')
for method_label, key in [('PC Replay (paper)', 'pc'), ('VAE Replay (paper)', 'vae')]:
    r = PAPER[key]
    print(f"{method_label:<22}"
          f" {pct(r['avg_accuracy']):>12}"
          f" {pct(r['avg_forgetting']):>12}"
          f" {'N/A':>10} {'N/A':>10}")

## 6. Bar Chart — Key Metrics vs Paper

In [ ]:
metrics_to_plot = [
    ('avg_accuracy',  'Average Accuracy (%)', True),
    ('avg_forgetting', 'Forgetting (%)',       False),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

bar_labels = ['PC (paper)', 'PC (ours)', 'VAE (paper)', 'VAE (ours)']
bar_colors = ['#2196F3', '#90CAF9', '#FF5722', '#FFAB91']
x = np.arange(len(bar_labels))

for ax, (metric_key, ylabel, higher_is_better) in zip(axes, metrics_to_plot):
    vals = [
        PAPER['pc'][metric_key],
        results['pc'][metric_key],
        PAPER['vae'][metric_key],
        results['vae'][metric_key],
    ]
    errs = [
        0, results['pc'][metric_key + '_std'],
        0, results['vae'][metric_key + '_std'],
    ]
    bars = ax.bar(x, [v * 100 for v in vals],
                  yerr=[e * 100 for e in errs],
                  color=bar_colors, capsize=5, edgecolor='white', linewidth=0.8)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val*100:.1f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(bar_labels, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 115)
    ax.set_title(ylabel, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    arrow = '↑ better' if higher_is_better else '↓ better'
    ax.set_xlabel(arrow, fontsize=9, labelpad=2)

fig.suptitle('Split-MNIST: ours (quick mode) vs paper reference', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Inspect Generated Replay Samples

We train generators on task 1 (digits 0 & 1) and sample from them to visualise replay quality.

In [ ]:
# Build and train a generator for task 0 only
x_task0, y_task0 = tasks[0][0], tasks[0][1]
classes_task0 = sorted(np.unique(y_task0).tolist())
print(f'Task 0 classes: {classes_task0}  n={len(x_task0)}')

def make_and_train_generator(method, seed):
    gen = clr._make_generator(
        method=method,
        data_dim=784,
        latent_dim=clr.DATASET_CONFIGS['mnist']['latent_dim'],
        hidden_dim=clr.DATASET_CONFIGS['mnist']['hidden_dim'],
        seed=seed,
    )
    gen.fit(x_task0,
            n_epochs=clr.DATASET_CONFIGS['mnist']['quick_epochs_gen'],
            batch_size=128, lr=1e-3, seed=seed)
    return gen

print('Training PC generator ...')
gen_pc  = make_and_train_generator('pc',  seed=SEED)
print('Training VAE generator ...')
gen_vae = make_and_train_generator('vae', seed=SEED)
print('Done.')

In [ ]:
N_SAMPLES = 12

rng = np.random.default_rng(SEED)
label_cycle = np.tile(classes_task0, N_SAMPLES // len(classes_task0) + 1)[:N_SAMPLES]

samples_pc  = gen_pc.generate(N_SAMPLES,  labels=label_cycle, seed=SEED)
samples_vae = gen_vae.generate(N_SAMPLES, labels=label_cycle, seed=SEED)

# Also pick N_SAMPLES real images for comparison
real_idx = rng.choice(len(x_task0), N_SAMPLES, replace=False)
real_imgs = x_task0[real_idx]

fig, axes = plt.subplots(3, N_SAMPLES, figsize=(N_SAMPLES * 1.2, 4))
row_labels = ['Real', 'PC replay', 'VAE replay']
for row, (imgs, row_label) in enumerate(zip([real_imgs, samples_pc, samples_vae], row_labels)):
    for col in range(N_SAMPLES):
        axes[row, col].imshow(imgs[col].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(row_label, fontsize=9, rotation=90, labelpad=4)

fig.suptitle('Task 1 images: real vs PC replay vs VAE replay', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# Pixel statistics
print(f"Real    — mean: {real_imgs.mean():.3f}   std: {real_imgs.std():.3f}   "
      f"range: [{real_imgs.min():.3f}, {real_imgs.max():.3f}]")
print(f"PC gen  — mean: {samples_pc.mean():.3f}   std: {samples_pc.std():.3f}   "
      f"range: [{samples_pc.min():.3f}, {samples_pc.max():.3f}]")
print(f"VAE gen — mean: {samples_vae.mean():.3f}   std: {samples_vae.std():.3f}   "
      f"range: [{samples_vae.min():.3f}, {samples_vae.max():.3f}]")

## 8. Notes on Deviations from the Paper

| Aspect | Paper | This implementation | Expected impact |
|--------|-------|--------------------|-----------------|
| Epochs/task (MNIST) | **50** | 10 (quick) / 50 (full) | Main driver of accuracy gap in quick mode |
| Generator strategy | Single generator, fine-tuned each task | **Per-task generators** trained on real data only | Avoids generator-level forgetting; requires more memory |
| CIFAR classifier | ResNet-18 | MLP (2×512) | Lower CIFAR accuracy |
| Independent trials | 5 | 1 (quick) / 5 (full) | Quick results have high variance |
| Framework | Not specified | FabricPC (JAX) + Flax | Different numerical paths but equivalent algorithms |

### To reproduce paper numbers

```python
QUICK    = False   # 50 epochs/task
N_TRIALS = 5       # match paper's 5-trial average
```

Or from the command line:

```bash
python examples/continual_learning_replay.py --dataset mnist --n_trials 5
```